In [49]:
print("Training process for Random Forest...\n\n🛢️ Random Forest Pipeline: SMOTE -> Random Forest")
rf_pipeline = ImbPipeline([
    (
        "smote", 
        SMOTE(random_state=42)
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])
print("✅ Done!\n")

print("🔎 Random Forest hyperparameter search...")
rf_params = {
    "model__n_estimators": randint(150, 700),
    "model__max_depth": [10, 15, 20, 25, 30, None],
    "model__min_samples_split": randint(5, 50),
    "model__min_samples_leaf": randint(2, 15),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}
print("✅ Done!\n")

print("🚀 Initializing cross-validation object...")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("✅ Done!\n")

print("🎲 Initializing a search manager object...")
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter = 30,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
print("✅ Done!\n")

Training process for Random Forest...

🛢️ Random Forest Pipeline: SMOTE -> Random Forest
✅ Done!

🔎 Random Forest hyperparameter search...
✅ Done!

🚀 Initializing cross-validation object...
✅ Done!

🎲 Initializing a search manager object...
✅ Done!



In [30]:
print("🎓 Fitting the search manager...")
rf_search.fit(X_train_rf, y_train_rf)

print(f"Best hyperparameters for Random Forest model are:\n{rf_search.best_params_}")
print(f"The best score is:{rf_search.best_score_}\n")

best_rf = rf_search.best_estimator_
print("💾 Saving best performing Random Forest model...")

try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(best_rf, f"../models/random_forest_best_{timestamp}.joblib")
    joblib.dump(rf_search, f"../models/rf_random_search_{timestamp}.joblib")
    print("✅ Random Forest model with best performance score was successfully saved!")
    
except:
    print("❌ Something went wrong during the model serialization...Check Traceback call")

🎓 Fitting the search manager...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best hyperparameters for Random Forest model are:
{'model__max_depth': 25, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 12, 'model__min_samples_split': 12, 'model__n_estimators': 170}
The best score is:0.9959988922878997

💾 Saving best performing Random Forest model...
✅ Random Forest model with best performance score was successfully saved!


In [28]:
def create_lof_pipeline(
    n_neighbors=50,
    contamination=0.05,
    metric="euclidean",
    p=2,
    leaf_size=30,
):
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "lof",
            LocalOutlierFactor(
                n_neighbors=n_neighbors,
                contamination=contamination,
                metric=metric,
                p=p,
                leaf_size=leaf_size,
                novelty=True,
                n_jobs=-1,
            ),
        ),
    ])

In [51]:
## 1. LOF Baseline

baseline_lof = create_lof_pipeline(
    n_neighbors=50,
    contamination=0.05,
    metric="euclidean",
    p=2,
    leaf_size=30,
)

baseline_lof.fit(X_train)

baseline_scores = -baseline_lof.score_samples(
    X_validation
)

baseline_predictions = pd.DataFrame({
    "txId": validation_df["txId"].values,
    "anomaly_score": baseline_scores,
})

baseline_eval = baseline_predictions.merge(
    validation_labels,
    on="txId",
    how="inner",
)

baseline_y_true = baseline_eval["is_illicit"].astype(int)
baseline_y_score = baseline_eval["anomaly_score"]

baseline_roc_auc = roc_auc_score(
    baseline_y_true,
    baseline_y_score,
)

baseline_pr_auc = average_precision_score(
    baseline_y_true,
    baseline_y_score,
)

print("Baseline LOF")
print("=" * 40)
print(f"ROC-AUC: {baseline_roc_auc:.6f}")
print(f"PR-AUC:  {baseline_pr_auc:.6f}")

Baseline LOF
ROC-AUC: 0.473827
PR-AUC:  0.076293


In [52]:
## 2. LOF Hyperparameter Search

n_neighbors_values = [
    75,
    100,
    125,
    150,
    175,
    200,
    250,
    300,
    350,
]

lof_search_results = []

for n_neighbors in n_neighbors_values:

    print(
        f"Testing n_neighbors={n_neighbors}"
    )

    lof_model = create_lof_pipeline(
        n_neighbors=n_neighbors,
        contamination=0.05,
        metric="euclidean",
        p=2,
        leaf_size=30,
    )

    lof_model.fit(X_train)

    anomaly_scores = -lof_model.score_samples(
        X_validation
    )

    search_predictions = pd.DataFrame({
        "txId": validation_df["txId"].values,
        "anomaly_score": anomaly_scores,
    })

    search_eval = search_predictions.merge(
        validation_labels,
        on="txId",
        how="inner",
    )

    y_true = search_eval["is_illicit"].astype(int)
    y_score = search_eval["anomaly_score"]

    lof_search_results.append({
        "n_neighbors": n_neighbors,
        "metric": "euclidean",
        "leaf_size": 30,
        "roc_auc": roc_auc_score(
            y_true,
            y_score,
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_score,
        ),
    })

lof_search_results_df = (
    pd.DataFrame(lof_search_results)
    .sort_values(
        by=["pr_auc", "roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(lof_search_results_df)

Testing n_neighbors=75
Testing n_neighbors=100
Testing n_neighbors=125
Testing n_neighbors=150
Testing n_neighbors=175
Testing n_neighbors=200
Testing n_neighbors=250
Testing n_neighbors=300
Testing n_neighbors=350


,n_neighbors,metric,leaf_size,roc_auc,pr_auc
0,125,euclidean,30,0.508838,0.105038
1,150,euclidean,30,0.492528,0.098484
2,100,euclidean,30,0.532648,0.098443
3,175,euclidean,30,0.481320,0.095613
4,200,euclidean,30,0.472817,0.093928
5,250,euclidean,30,0.463485,0.091758
6,300,euclidean,30,0.457451,0.090182
7,350,euclidean,30,0.457143,0.089772
8,75,euclidean,30,0.506001,0.083006


In [53]:
## 3. Selected LOF Configuration

best_lof_params = lof_search_results_df.iloc[0]

print("Selected configuration:")
print(
    f"n_neighbors: {int(best_lof_params['n_neighbors'])}"
)
print(
    f"metric:      {best_lof_params['metric']}"
)
print(
    f"leaf_size:   {int(best_lof_params['leaf_size'])}"
)
print(
    f"ROC-AUC:     {best_lof_params['roc_auc']:.6f}"
)
print(
    f"PR-AUC:      {best_lof_params['pr_auc']:.6f}"
)

Selected configuration:
n_neighbors: 125
metric:      euclidean
leaf_size:   30
ROC-AUC:     0.508838
PR-AUC:      0.105038


In [54]:
## 4. Selected LOF — Validation Evaluation
selected_lof = create_lof_pipeline(
    n_neighbors=int(best_lof_params["n_neighbors"]),
    contamination=0.05,
    metric=best_lof_params["metric"],
    p=2,
    leaf_size=int(best_lof_params["leaf_size"]),
)

selected_lof.fit(X_train)

print("Selected LOF trained.")
print("Training shape:", X_train.shape)

validation_scores = -selected_lof.score_samples(
    X_validation
)

validation_predictions = pd.DataFrame({
    "txId": validation_df["txId"].values,
    "time_step": validation_df["time_step"].values,
    "anomaly_score": validation_scores,
})

validation_eval = validation_predictions.merge(
    validation_labels,
    on="txId",
    how="inner",
)

print(
    "Validation transactions:",
    len(validation_predictions)
)

print(
    "Labeled validation transactions:",
    len(validation_eval)
)

display(validation_eval.head())

y_val_true = validation_eval["is_illicit"].astype(int)
y_val_score = validation_eval["anomaly_score"]

selected_roc_auc = roc_auc_score(
    y_val_true,
    y_val_score,
)

selected_pr_auc = average_precision_score(
    y_val_true,
    y_val_score,
)

print("Selected LOF validation performance")
print("=" * 45)
print(f"ROC-AUC: {selected_roc_auc:.6f}")
print(f"PR-AUC:  {selected_pr_auc:.6f}")

Selected LOF trained.
Training shape: (136265, 165)
Validation transactions: 30680
Labeled validation transactions: 7829


,txId,time_step,anomaly_score,is_illicit
0,54751137,35,2.485928,0
1,67576672,35,1.040865,0
2,69767012,35,1.578661,0
3,70384401,35,1.095432,1
4,67603017,35,1.250184,1


Selected LOF validation performance
ROC-AUC: 0.508838
PR-AUC:  0.105038


In [76]:
## 5. Final LOF Model

# ============================================================
# FINAL LOF MODEL
# ============================================================

# Use the selected hyperparameters from validation search.
final_lof = create_lof_pipeline(
    n_neighbors=int(best_lof_params["n_neighbors"]),
    contamination=0.05,
    metric=best_lof_params["metric"],
    p=2,
    leaf_size=int(best_lof_params["leaf_size"]),
)

# Historical data available before the test period:
# time steps 1–41
X_train_lof_final = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] <= 41
    ].drop(columns=["txId", "time_step"])
    .copy()
)

print("Final LOF training data:", X_train_lof_final.shape)

# Fit the complete pipeline.
#
# StandardScaler is fitted ONLY here on time steps 1–41,
# then LOF is fitted on the scaled data.
final_lof.fit(X_train_lof_final)

print("Final LOF trained successfully.")

Final LOF training data: (166945, 165)
Final LOF trained successfully.


In [77]:
## 7. Save Final LOF Model

model_path = Path("models/lof/lof_final.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_lof, model_path)
print(f"Final LOF saved to: {model_path}")

Final LOF saved to: models/lof/lof_final.joblib


In [82]:
# ============================================================
# CREATE TEST LABELS FOR TEMPORAL EVALUATION
# ============================================================

test_labels = (
    base_df[
        (base_df["class"] != "unknown") &
        (base_df["time_step"] >= 42)
    ][
        ["txId"]
    ]
    .copy()
)

test_labels["is_illicit"] = (
    base_df[
        (base_df["class"] != "unknown") &
        (base_df["time_step"] >= 42)
    ]["class"]
    == "1"
).astype(int)


print("Test labels:", test_labels.shape)
print(test_labels["is_illicit"].value_counts())

Test labels: (8841, 2)
is_illicit
0    8433
1     408
Name: count, dtype: int64


In [83]:
# ============================================================
# FINAL LOF — TEMPORAL TEST EVALUATION
# ============================================================

X_test_lof_final = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] >= 42
    ].drop(columns=["txId", "time_step"])
    .copy()
)


test_lof_scores = -final_lof.score_samples(
    X_test_lof_final
)


test_lof_predictions = pd.DataFrame({
    "txId": unsupervised_full_df[
        unsupervised_full_df["time_step"] >= 42
    ]["txId"].values,

    "time_step": unsupervised_full_df[
        unsupervised_full_df["time_step"] >= 42
    ]["time_step"].values,

    "anomaly_score": test_lof_scores,
})

test_lof_eval = test_lof_predictions.merge(
    test_labels,
    on="txId",
    how="inner",
)


y_test_true = test_lof_eval["is_illicit"].astype(int)

y_test_score = test_lof_eval["anomaly_score"]


print("=" * 60)
print("FINAL LOF — TEMPORAL TEST")
print("=" * 60)

print(
    f"ROC-AUC: {roc_auc_score(y_test_true, y_test_score):.6f}"
)

print(
    f"PR-AUC:  {average_precision_score(y_test_true, y_test_score):.6f}"
)

FINAL LOF — TEMPORAL TEST
ROC-AUC: 0.460650
PR-AUC:  0.047858


In [108]:
# ============================================================
# STORE TEMPORAL LOF RESULTS
# ============================================================

temporal_lof_roc = roc_auc_score(
    y_test_true,
    y_test_score
)

temporal_lof_pr = average_precision_score(
    y_test_true,
    y_test_score
)


print("Temporal LOF stored:")
print("ROC-AUC:", temporal_lof_roc)
print("PR-AUC:", temporal_lof_pr)

Temporal LOF stored:
ROC-AUC: 0.2808789233706052
PR-AUC: 0.029305052349612188


In [85]:
# ============================================================
# PREPARE TEMPORAL IF DATASETS
# ============================================================

X_train_if = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] <= 34
    ]
    .drop(
        columns=["txId", "time_step"]
    )
    .copy()
)


X_validation_if = (
    unsupervised_full_df[
        (unsupervised_full_df["time_step"] >= 35) &
        (unsupervised_full_df["time_step"] <= 41)
    ]
    .drop(
        columns=["txId", "time_step"]
    )
    .copy()
)


print("X_train_if:", X_train_if.shape)
print("X_validation_if:", X_validation_if.shape)

X_train_if: (136265, 165)
X_validation_if: (30680, 165)


In [86]:
print("📚 Train Isolation Forest on training period (time_step 1-34)\n")
if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


if_model.fit(X_train_if)


print("🌱 Isolation Forest trained")

📚 Train Isolation Forest on training period (time_step 1-34)

🌱 Isolation Forest trained


In [87]:
print("💡 Create prediction helper function")
def create_if_predictions(
    model,
    X,
    df
):
    """
    Generate Isolation Forest outputs.
    """
    
    predictions = pd.DataFrame({
        "txId": df["txId"].values,
        "time_step": df["time_step"].values,

        # sklearn score:
        # higher = more normal
        # lower = more abnormal
        "anomaly_score": -model.score_samples(X),

        # sklearn decision:
        # negative = anomaly
        # positive = normal
        "decision_function": model.decision_function(X),

        # binary output
        "prediction": (
            model.predict(X) == -1
        ).astype(int)
    })


    return predictions

print("🆕 Prediction helper function created!")

💡 Create prediction helper function
🆕 Prediction helper function created!


In [91]:
# ============================================================
# PREPARE VALIDATION METADATA FOR IF PREDICTIONS
# ============================================================

validation_if_df = (
    unsupervised_full_df[
        (unsupervised_full_df["time_step"] >= 35) &
        (unsupervised_full_df["time_step"] <= 41)
    ]
    .copy()
)


print(
    "validation_if_df:",
    validation_if_df.shape
)

validation_if_df.head()

# ============================================================
# PREPARE VALIDATION LABELS FOR IF EVALUATION
# ============================================================

validation_labels_if = (
    random_forest_df[
        (random_forest_df["time_step"] >= 35) &
        (random_forest_df["time_step"] <= 41)
    ][
        ["txId", "is_illicit"]
    ]
    .copy()
)


print(
    "Validation labels:",
    validation_labels_if.shape
)

validation_labels_if.head()

validation_if_df: (30680, 167)
Validation labels: (7829, 2)


,txId,is_illicit
136276,54751137,0
136277,67576672,0
136278,69767012,0
136279,70384401,1
136280,67603017,1


In [92]:
val_predictions_df = create_if_predictions(
    if_model,
    X_validation_if,
    validation_if_df
)


val_predictions_df.head()
print("🔄 Generated validation predictions\n")

val_eval_df = val_predictions_df.merge(
    validation_labels_if,
    on="txId",
    how="inner"
)


print(
    "🕵🏻 Validation labeled transactions:",
    len(val_eval_df)
)

val_eval_df.head()
print("✅ Created validation evaluation dataset")

🔄 Generated validation predictions

🕵🏻 Validation labeled transactions: 7829
✅ Created validation evaluation dataset


In [94]:
y_true = val_eval_df["is_illicit"]

y_score = val_eval_df["anomaly_score"]


print("==============================")
print("Validation IF ranking")
print("==============================")


print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_score
    )
)


print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_score
    )
)

Validation IF ranking
ROC-AUC: 0.09292309922446909
PR-AUC: 0.045936296581052076


In [95]:
print("👩🏻‍🍳 Prepare final training data (time_step 1-41)")
X_train_if_full = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] <= 41
    ]
    .drop(
        columns=["txId", "time_step"]
    )
)


print(
    "🏁 Final training shape:",
    X_train_if_full.shape
)


👩🏻‍🍳 Prepare final training data (time_step 1-41)
🏁 Final training shape: (166945, 165)


In [101]:
# ============================================================
# DEFINE UNSUPERVISED FEATURE COLUMNS
# ============================================================

feature_columns_unsupervised = [
    col
    for col in unsupervised_full_df.columns
    if col not in ["txId", "time_step"]
]


print(
    "Number of unsupervised features:",
    len(feature_columns_unsupervised)
)

Number of unsupervised features: 165


In [104]:
# ============================================================
# PREPARE TEMPORAL TEST DATA FOR IF
# ============================================================

test_if_df = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] >= 42
    ]
    .copy()
)


X_test_if = (
    test_if_df[
        feature_columns_unsupervised
    ]
    .copy()
)

# ============================================================
# PREPARE TEST LABELS
# ============================================================

test_labels_if = (
    random_forest_df[
        random_forest_df["time_step"] >= 42
    ][
        ["txId", "is_illicit"]
    ]
    .copy()
)


print("test_labels_if:", test_labels_if.shape)

print("test_if_df:", test_if_df.shape)
print("X_test_if:", X_test_if.shape)

test_labels_if: (8841, 2)
test_if_df: (36824, 167)
X_test_if: (36824, 165)


In [105]:
final_if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


final_if_model.fit(
    X_train_if_full
)

joblib.dump(final_if_model, "models/isolation_forest/isolation_forest_best.joblib")
print("👩🏻‍💻📓✍🏻💡 Final IF trained on time steps 1-41!")

# ============================================================
# GENERATE FINAL IF TEST PREDICTIONS
# ============================================================

test_predictions_df = create_if_predictions(
    final_if_model,
    X_test_if,
    test_if_df
)


print(
    "Test transactions predicted:",
    len(test_predictions_df)
)

test_predictions_df.head()

print("🖇️ Merge test labels for evaluation\n")
test_eval_df = test_predictions_df.merge(
    test_labels_if,
    on="txId",
    how="inner"
)


print(
    "🧾 Test labeled transactions:",
    len(test_eval_df)
)

👩🏻‍💻📓✍🏻💡 Final IF trained on time steps 1-41!
Test transactions predicted: 36824
🖇️ Merge test labels for evaluation

🧾 Test labeled transactions: 8841


In [106]:
# ============================================================
# FINAL IF — TEMPORAL TEST EVALUATION
# ============================================================

X_test_if_final = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] >= 42
    ]
    .drop(
        columns=["txId", "time_step"]
    )
)


test_if_scores = -final_if_model.score_samples(
    X_test_if_final
)


test_if_predictions = pd.DataFrame({
    "txId": test_if_df["txId"].values,
    "time_step": test_if_df["time_step"].values,
    "anomaly_score": test_if_scores,
})


test_if_eval = test_if_predictions.merge(
    test_labels_if,
    on="txId",
    how="inner",
)


y_test_true = test_if_eval["is_illicit"].astype(int)

y_test_score = test_if_eval["anomaly_score"]


print("=" * 60)
print("FINAL IF — TEMPORAL TEST")
print("=" * 60)

print(
    f"ROC-AUC: {roc_auc_score(y_test_true, y_test_score):.6f}"
)

print(
    f"PR-AUC:  {average_precision_score(y_test_true, y_test_score):.6f}"
)

FINAL IF — TEMPORAL TEST
ROC-AUC: 0.280879
PR-AUC:  0.029305


In [115]:
# ============================================================
# STORE TEMPORAL IF RESULTS
# ============================================================

temporal_if_roc = roc_auc_score(
    test_eval_df["is_illicit"],
    -test_eval_df["anomaly_score"]
)

temporal_if_pr = average_precision_score(
    test_eval_df["is_illicit"],
    -test_eval_df["anomaly_score"]
)

print(f"ROC-AUC: {temporal_if_roc:.6f}")
print(f"PR-AUC:  {temporal_if_pr:.6f}")

ROC-AUC: 0.719121
PR-AUC:  0.287346


In [117]:
results = pd.DataFrame([
    {
        "model": "LOF",
        "experiment": "Temporal",
        "roc_auc": temporal_lof_roc,
        "pr_auc": temporal_lof_pr
    },
    {
        "model": "Isolation Forest",
        "experiment": "Temporal",
        "roc_auc": temporal_if_roc,
        "pr_auc": temporal_if_pr
    },
    {
        "model": "LOF",
        "experiment": "Random label holdout",
        "roc_auc": random_lof_roc_auc,
        "pr_auc": random_lof_pr_auc
    },
    {
        "model": "Isolation Forest",
        "experiment": "Random label holdout",
        "roc_auc": random_if_roc_auc,
        "pr_auc": random_if_pr_auc
    },
])


display(results)

results.to_csv(
    "models/unsupervised_model_comparison.csv",
    index=False
)

print("Saved.")

,model,experiment,roc_auc,pr_auc
0,LOF,Temporal,0.280879,0.029305
1,Isolation Forest,Temporal,0.719121,0.287346
2,LOF,Random label holdout,0.227117,0.058514
3,Isolation Forest,Random label holdout,0.905277,0.659699


Saved.
